In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# DUMMY SUBMISSION:

In [ ]:
# import pandas as pd

# submission_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# submission_df.to_csv("submission.csv",index=False)

# IMPORTING ALL DEPENDENCIES:

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


# LOADING ALL DATASETS:

In [ ]:
train_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
submission_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

# EXPLONATORY DATA ANALYSIS:

#  DISPLAY FIRST FIVE ROWS OF TRAIN_DF:

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df.info()

In [ ]:
print("#" * 50)
print(f"TRAIN_DF SHAPE: {train_df.shape}")
print(f"TEST_DF SHAPE: {test_df.shape}")

print("#" * 50)
print(f"TRAIN_DF COLUMNS: {train_df.columns}")
print(f"TEST_DF COLUMNS: {test_df.columns}")


# Target class distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='answer', data=train_df, order=['A', 'B', 'C', 'D', 'E'])
plt.title("Distribution of Correct Answers")
plt.xlabel("Answer Class")
plt.ylabel("Count")
plt.show()

# IDENTIFY AND HANDLE DUPLICATES

In [ ]:
duplicate_count = train_df.duplicated(subset=['prompt']).sum()
print(f"Duplicate prompts in train dataset: {duplicate_count}")

train_df = train_df.drop_duplicates(subset=['prompt']).reset_index(drop=True)
print(f"Cleaned dataset size : {len(train_df)}")

# IDENTIFY AND HANDLE MISSING VALUES

In [ ]:
print("#" * 50)
print(f"MISSING VALUE IN TRAIN_DF:{train_df.isnull().sum()}")
print("#" * 50)
print(f"MISSING VALUE IN TEST_DF:{test_df.isnull().sum()}")

columns = ["prompt", "A", "B", "C", "D", "E"]
for column in columns:
    train_df[column] = train_df[column].fillna("")
    if column in test_df.columns:
        test_df[column] = test_df[column].fillna("")

# TEXT CLEANING:

In [ ]:
# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

In [ ]:
# for col in columns:
#     train_df[col] = train_df[col].apply(clean_text)
#     if col in test_df.columns:
#         test_df[col] = test_df[col].apply(clean_text)

In [ ]:
train_df.head()

# TF-IDF VECTORIZER | COSINE SIMILARITIES | MAP@3

In [ ]:
all_columns=['prompt','A','B','C','D','E']
combined_text_of_all_columns = train_df[all_columns].fillna('').astype(str).agg(' '.join, axis=1)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english",ngram_range=(1,2), max_features=1000)
vectorizer.fit(combined_text_of_all_columns)

In [ ]:
scores = []

for index, row in train_df.iterrows():
    prompt_text = row['prompt']
    actual_answer_text = row['answer']
    
    prompt_vector = vectorizer.transform([prompt_text])
    
    # Calculate similarity for each of the 5 options
    similarities = {}
    for option in ['A', 'B', 'C', 'D', 'E']:
        option_text = row[option]
        option_vector = vectorizer.transform([option_text])
        
        score = cosine_similarity(prompt_vector, option_vector)[0][0]
        similarities[option] = score
        
    # print(f"INDEX : {index}") 
    # print(f"ACTUAL ANSWER : {actual_answer_text}")
    # print(f"COSINE SIMILARITIES : {similarities}") 
    
    sorted_options = sorted(similarities, key=similarities.get, reverse=True)
    # print(f"SORTED_OPTIONS : {sorted_options}")
    
    top_3 = sorted_options[:3]
    # print(f"TOP3 VALUES : {top_3}")
    
    if index <100:
        print(f"Index: {index} | Actual: {actual_answer_text} | Predicted Top3: {top_3}")
    
    if actual_answer_text == top_3[0]:
        scores.append(1.0)
    elif actual_answer_text == top_3[1]:
        scores.append(0.5)
    elif actual_answer_text == top_3[2]:
        scores.append(1/3)
    else:
        scores.append(0.0)

final_map3 = np.mean(scores)
round(final_map3 ,4)

# WANDB LOGIN 

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [ ]:
import wandb

wandb.login(key=WB_KEY)

# MODEL1 : TF-IDF + LOGISTIC REGRESSION

# SPLITTING OF DATASET

In [ ]:
# train_split, val_split = train_test_split(
#     train_df, 
#     test_size=0.2, 
#     random_state=42, 
#     stratify=train_df['answer']
# )

# INITIALIZE WANDB RUN

In [ ]:
# hyperparameters = {
#     "model_type": "TFIDF_Logistic_Regression",
#     "word_max_features": 40000,
#     "char_max_features": 60000,
#     "word_ngram_range": (1, 2),
#     "char_ngram_range": (3, 5),
#     "C_regularization": 1.5,
#     "max_iter": 1000,
#     "solver": "liblinear"
# }

# wandb.init(
#     project="23f2002113-t22026",
#     name="tfidf-logistic-regression",
#     config=hyperparameters
# )

In [ ]:
# def process_ranking_data(df, is_test=False):
#     texts = []
#     labels = []
#     options_list = ['A', 'B', 'C', 'D', 'E']
    
#     for _, row in df.iterrows():
#         prompt = row['prompt']
#         for opt in options_list:
#             # Pair prompt with option text
#             combined_text = f"{prompt} [SEP] {row[opt]}"
#             texts.append(combined_text)
            
#             if not is_test:
#                 labels.append(1 if row['answer'] == opt else 0)
                
#     return texts, np.array(labels) if not is_test else None

# train_texts, train_labels = process_ranking_data(train_split, is_test=False)
# val_texts, val_labels = process_ranking_data(val_split, is_test=False)
# test_texts, _ = process_ranking_data(test_df, is_test=True)


# # DUAL TF-IDF FEATURE EXTRACTION
# word_vectorizer = TfidfVectorizer(
#     ngram_range=tuple(wandb.config.word_ngram_range), 
#     max_features=wandb.config.word_max_features, 
#     sublinear_tf=True, 
#     stop_words='english'
# )

# char_vectorizer = TfidfVectorizer(
#     analyzer='char', 
#     ngram_range=tuple(wandb.config.char_ngram_range), 
#     max_features=wandb.config.char_max_features, 
#     sublinear_tf=True
# )

# # Fit vectorizers on training splits
# word_vectorizer.fit(train_texts)
# char_vectorizer.fit(train_texts)

# # Combine representations
# X_train = hstack([word_vectorizer.transform(train_texts), char_vectorizer.transform(train_texts)])
# X_val = hstack([word_vectorizer.transform(val_texts), char_vectorizer.transform(val_texts)])
# X_test = hstack([word_vectorizer.transform(test_texts), char_vectorizer.transform(test_texts)])

# #Training regression model
# model = LogisticRegression(
#     C=wandb.config.C_regularization, 
#     max_iter=wandb.config.max_iter, 
#     solver=wandb.config.solver
# )
# model.fit(X_train, train_labels)

# val_preds = model.predict_proba(X_val)[:, 1]

# # Reshape back into (Questions, 5 Options)
# num_val_questions = len(val_split)
# val_scores_matrix = val_preds.reshape(num_val_questions, 5)

# map3_scores = []
# options_list = ['A', 'B', 'C', 'D', 'E']
# val_split = val_split.reset_index(drop=True)

# for i in range(num_val_questions):
#     sorted_option_indices = np.argsort(-val_scores_matrix[i])
#     top_3_predictions = [options_list[idx] for idx in sorted_option_indices[:3]]
#     correct_answer = val_split.loc[i, 'answer']
    
#     if correct_answer == top_3_predictions[0]:
#         map3_scores.append(1.0)
#     elif correct_answer == top_3_predictions[1]:
#         map3_scores.append(0.5)
#     elif correct_answer == top_3_predictions[2]:
#         map3_scores.append(1/3)
#     else:
#         map3_scores.append(0.0)

# print(f" Validation MAP@3 Score: {np.mean(map3_scores):.4f} ")

In [ ]:
# from sklearn.metrics import accuracy_score, f1_score

# val_preds_binary = (val_preds >= 0.5).astype(int)

# accuracy = accuracy_score(val_labels, val_preds_binary)
# f1 = f1_score(val_labels, val_preds_binary, average='binary')

# print(f"\n--- EVALUATION RESULTS ---")
# print(f"Validation Accuracy: {accuracy:.4f}")
# print(f"Validation F1-Score: {f1:.4f}")
# print(f"Validation MAP@3 Score: {final_map3:.4f}")

# #WANDB LOGS 
# wandb.log({
#     "val_accuracy": accuracy,
#     "val_f1_score": f1,
#     "val_map3": final_map3
# })

# # Close the WandB run session
# wandb.finish()

In [ ]:
# #final predictions on test dataset
# test_preds = model.predict_proba(X_test)[:, 1]
# num_test_questions = len(test_df)
# test_scores_matrix = test_preds.reshape(num_test_questions, 5)

# submission_predictions = []
# for i in range(num_test_questions):
#     sorted_option_indices = np.argsort(-test_scores_matrix[i])
#     top_3_predictions = [options_list[idx] for idx in sorted_option_indices[:3]]
#     submission_predictions.append(" ".join(top_3_predictions))

# submission_df = pd.DataFrame({
#     'id': test_df['id'] if 'id' in test_df.columns else test_df.index,
#     'Prediction': submission_predictions
# })

# submission_df.to_csv("submission.csv", index=False)

# MODEL 2-PRETRAINED MODEL(DISTILBERT-BASE-UNCASED)

# IMPORT LIBRARIES

In [ ]:
import gc

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from torch.optim import AdamW


# Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# SPLIT DATA INTO TRAIN AND VALIDATION

In [ ]:
train_split, validation_split = train_test_split(
    train_df, 
    train_size=0.8, 
    random_state=42, 
    stratify=train_df['answer']
)

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=224, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']
        self.labels_map = {option: index for index, option in enumerate(self.options)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        prompt = row['prompt']
        
        # We pair the prompt with all 5 options individually
        prompts = [prompt] * 5
        choices = [str(row[option]) for option in self.options]
        
        # Convert text pairs into sequence arrays
        features = self.tokenizer(
            prompts,
            choices,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors="pt"
        )
        
        # Format tensors so PyTorch can bundle them into batches
        item = {key: value.squeeze(0) for key, value in features.items()}
        
        if not self.is_test:
            # Map correct string answer (e.g., 'B') to numerical index (e.g., 1)
            item['label'] = torch.tensor(self.labels_map[row['answer']], dtype=torch.long)
            
        return item

# CONFIGURATION

In [ ]:
# HYPERPARAMETERS
hyperparameters = {
    "model_name": "distilbert-base-uncased",
    "learning_rate": 2e-5,
    "batch_size": 4,  
    "epochs": 3,
    "max_len": 224
}

# Initialize your Weights & Biases 
wandb.init(
    project="23f2002113-t22026",
    name="distilbert-mcq",
    config=hyperparameters
)


# import tokenizer and  pretrained multiple-choice model
tokenizer = AutoTokenizer.from_pretrained(wandb.config["model_name"])
model = AutoModelForMultipleChoice.from_pretrained(wandb.config["model_name"]).to(device)

# PyTorch DataLoaders (it converts our data into batches of size 4)
train_dataset = MCQDataset(train_split, tokenizer, max_len=wandb.config["max_len"])
validation_dataset = MCQDataset(validation_split, tokenizer, max_len=wandb.config["max_len"])

train_loader = DataLoader(train_dataset, batch_size=wandb.config["batch_size"], shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=wandb.config["batch_size"], shuffle=False)

# Apply optimizer to adjust weights during training
optimizer = AdamW(model.parameters(), lr=wandb.config["learning_rate"])

# MODEL TRAINING AND VALIDATION PHASE

In [ ]:
best_map3 = 0.0

for epoch in range(wandb.config["epochs"]):
    
    # --- TRAINING PHASE ---
    model.train()
    total_train_loss = 0.0
    
    for batch in train_loader:
        optimizer.zero_grad()
        
        # Send numerical batch IDs to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass through model to get loss and scores
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # Backward pass (calculates changes) and optimizer step (applies changes)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    
    # --- VALIDATION PHASE (EVALUATION) ---
    model.eval()
    total_validation_loss = 0.0
    validation_predictions = []
    validation_targets = []
    
    with torch.no_grad(): 
        for batch in validation_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_validation_loss += outputs.loss.item()
            
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            
            validation_predictions.extend(probs)
            validation_targets.extend(labels.cpu().numpy())
            
    avg_validation_loss = total_validation_loss / len(validation_loader)
    
    # Calculate MAP@3 metric for evaluation
    map3_scores = []
    for probs, correct_idx in zip(validation_predictions, validation_targets):
        top_3_indices = np.argsort(-probs)[:3] # descending order sort
        
        if correct_idx == top_3_indices[0]:
            map3_scores.append(1.0)
        elif correct_idx == top_3_indices[1]:
            map3_scores.append(0.5)
        elif correct_idx == top_3_indices[2]:
            map3_scores.append(1/3)
        else:
            map3_scores.append(0.0)
            
    epoch_map3 = np.mean(map3_scores)
    
    print(f"Epoch {epoch+1}/{wandb.config['epochs']} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_validation_loss:.4f} | Val MAP@3: {epoch_map3:.4f}")
    
    # If this epoch has the best validation score, save it
    if epoch_map3 > best_map3:
        best_map3 = epoch_map3
        torch.save(model.state_dict(), "best_distilbert_model.pt")
        print("--> Saved best model checkpoint.")
        
    # Log progress to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_validation_loss,
        "val_map3": epoch_map3
    })

wandb.finish()

# INFERENCE AND SUBMISSION PART

In [ ]:
# Load our saved best performing model 
if os.path.exists("best_distilbert_model.pt"):
    model.load_state_dict(torch.load("best_distilbert_model.pt"))
model.to(device)
model.eval()

# Prepare test datasets

MAX_LEN = 224
BATCH_SIZE = 4

test_dataset = MCQDataset(test_df, tokenizer, max_len=MAX_LEN, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_predictions_ranked = []
options_list = ['A', 'B', 'C', 'D', 'E']

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        
        for p in probs:
            top_3_indices = np.argsort(-p)[:3]
            top_3_labels = [options_list[idx] for idx in top_3_indices]
            test_predictions_ranked.append(" ".join(top_3_labels))

# Prepare the final submission dataframe
submission_df = pd.DataFrame({
    'id': test_df['id'] if 'id' in test_df.columns else test_df.index,
    'Prediction': test_predictions_ranked
})

# Save output to submission.csv 
submission_df.to_csv("submission.csv", index=False)